HW1 (Foundations - Ch. 2–3):
Normalization & tokenization: select a data set of your choice, conduct a brief ex-
ploratory data analysis, decide an arbitrary task you are supposed to implement, and
apply the text normalization and tokenization as discussed in the classed.
Implement either Edit Distance or Byte-pairing algorithm and apply it to you data set
and analysis the output and comment on it.

In [9]:
import re
from typing import List
import zipfile
import os
import pandas as pd
from itertools import combinations
from tqdm import tqdm

In [10]:
#!/bin/bash
!curl -L -o ~/Downloads/imdb-dataset-of-50k-movie-reviews.zip\
  https://www.kaggle.com/api/v1/datasets/download/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 25.7M  100 25.7M    0     0  7483k      0  0:00:03  0:00:03 --:--:-- 10.2M


In [11]:
os.makedirs("dataset_imdb", exist_ok=True)


In [12]:
zip_path = os.path.expanduser('~/Downloads/imdb-dataset-of-50k-movie-reviews.zip')
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('dataset_imdb')

In [13]:
imdb_data = pd.read_csv("dataset_imdb/IMDB Dataset.csv")

In [14]:
w = " ".join(imdb_data['review'].fillna('').astype(str))

In [15]:
def tokenize(text: str, pattern: str | None = None) -> List[str]:
    text = text.lower()
    pattern = r"(?:\w+')\w+|(?:[A-Z]\.)+|\w+(?:-\w+)*|[\w+\.]"
    tokens = re.findall(pattern, text)
    return tokens

In [16]:
tokenized_data = tokenize(w)

In [17]:
freq = dict()
vocab = set()

for word in tokenized_data:
    vocab.update(word)
    freq[word] = freq.get(word, 0) + 1

In [18]:
words = {tuple(word): count for word, count in freq.items()}

tmp_dict = dict()
for symb, count in words.items():
    for i in range(len(symb) - 1):
        pair = (symb[i], symb[i + 1])
        tmp_dict[pair] = tmp_dict.get(pair, 0) + count

In [19]:
max_pair = max(tmp_dict, key=tmp_dict.get)
new_words = dict()
for symb, count in words.items():
    i = 0
    new_symb = list()
    while i < len(symb):
        if i < len(symb) - 1 and (symb[i], symb[i + 1]) == max_pair:
            new_symb.append(symb[i] + symb[i + 1])
            i += 2
        else:
            new_symb.append(symb[i])
            i += 1
    new_words[tuple(new_symb)] = count

vocab.add(max_pair[0] + max_pair[1])
words = new_words

In [20]:
# добавит ьсюла _ или \w
def bpe(data, count_total: int):
    freq = dict()
    vocab = set()

    for word in data:
        vocab.update(word)
        freq[word] = freq.get(word, 0) + 1

    words = {tuple(word): count for word, count in freq.items()}

    for k in tqdm(range(count_total)):
        tmp_dict = dict()
        for symb, count in words.items():
            for i in range(len(symb) - 1):
                pair = (symb[i], symb[i + 1])
                tmp_dict[pair] = tmp_dict.get(pair, 0) + count

        max_pair = max(tmp_dict, key=tmp_dict.get)
        new_words = dict()
        for symb, count in words.items():
            i = 0
            new_symb = list()
            while i < len(symb):
                if i < len(symb) - 1 and (symb[i], symb[i + 1]) == max_pair:
                    new_symb.append(symb[i] + symb[i + 1])
                    i += 2
                else:
                    new_symb.append(symb[i])
                    i += 1
            new_words[tuple(new_symb)] = count

        vocab.add(max_pair[0] + max_pair[1])
        words = new_words

    return vocab

In [21]:
len_unique = len(set(tokenized_data[:12000]))
print(f'Unique size of vocab (without BPE): {len_unique}')

Unique size of vocab (without BPE): 3034


In [22]:
len_stast = len(set(char for word in tokenized_data[:25000] for char in word))
print(f'Vocab. size in the 0 iter of BPE: {len_stast}')

Vocab. size in the 0 iter of BPE: 45


In [23]:
bpe_result = bpe(tokenized_data[:25000], 300)
print(f'Vocab. size in the 300 iter of BPE: {len(bpe_result)}')
print('examples')
len(bpe_result)
for word in bpe_result:
    if len(word) > 2:
        print(word)

100%|██████████| 300/300 [00:01<00:00, 282.64it/s]

Vocab. size in the 300 iter of BPE: 345
examples
were
are
ast
ind
movies
time
bec
see
had
ick
our
tim
ould
would
oun
ate
his
n't
red
ment
what
ity
play
old
ive
fir
only
tor
how
com
kill
movi
way
out
and
other
her
don
tory
ther
who
there
ting
per
ong
him
fil
not
ies
lot
some
ite
pro
will
have
ere
low
ile
for
watch
ally
really
ble
their
been
ves
then
ill
wor
int
even
its
just
you
ant
can
ack
act
ough
ing
char
most
movie
off
ade
was
first
very
con
bad
good
ars
ell
the
ard
ist
ple
than
bet
end
wat
ome
ous
able
ain
ver
art
ent
this
ause
ine
where
much
ation
ter
get
ever
but
der
that
has
which
own
though
ber
ving
look
more
she
ust
reat
ich
about
could
one
mar
urn
story
est
any
ice
ound
ion
ace
film
now
uch
like
character
king
tain
into
from
with
they
life
does
man
dis
ence
ure
when
all
ght
ers
pre
them
ame
charac
scen
ide
thing
ess
it's
ful


In [24]:
tokenized_data[:12000]

['one',
 'of',
 'the',
 'other',
 'reviewers',
 'has',
 'mentioned',
 'that',
 'after',
 'watching',
 'just',
 '1',
 'oz',
 'episode',
 "you'll",
 'be',
 'hooked',
 '.',
 'they',
 'are',
 'right',
 'as',
 'this',
 'is',
 'exactly',
 'what',
 'happened',
 'with',
 'me',
 '.',
 'br',
 'br',
 'the',
 'first',
 'thing',
 'that',
 'struck',
 'me',
 'about',
 'oz',
 'was',
 'its',
 'brutality',
 'and',
 'unflinching',
 'scenes',
 'of',
 'violence',
 'which',
 'set',
 'in',
 'right',
 'from',
 'the',
 'word',
 'go',
 '.',
 'trust',
 'me',
 'this',
 'is',
 'not',
 'a',
 'show',
 'for',
 'the',
 'faint',
 'hearted',
 'or',
 'timid',
 '.',
 'this',
 'show',
 'pulls',
 'no',
 'punches',
 'with',
 'regards',
 'to',
 'drugs',
 'sex',
 'or',
 'violence',
 '.',
 'its',
 'is',
 'hardcore',
 'in',
 'the',
 'classic',
 'use',
 'of',
 'the',
 'word',
 '.',
 'br',
 'br',
 'it',
 'is',
 'called',
 'oz',
 'as',
 'that',
 'is',
 'the',
 'nickname',
 'given',
 'to',
 'the',
 'oswald',
 'maximum',
 'security',